# **Comprehensive Guide to RAG Architectures**

This notebook explores different patterns in Retrieval-Augmented Generation (RAG) architectures:
1. Basic Naive RAG Pipeline
2. Hybrid Search RAG (Vector + Keyword)
3. RAG with Reranking (Cross-Encoder)
4. Query Expansion / Transformative RAG
5. Agentic RAG
6. GraphRAG Architecture

**Prerequisites:** Install dependencies before running:
```bash
pip install langchain langchain-community langchain-chroma sentence-transformers faiss-cpu pandas matplotlib chromadb python-dotenv
```

In [ ]:
# Setup: Load environment variables from .env file
import os
import numpy as np

# Try to load dotenv, handle gracefully if not installed
try:
    from dotenv import load_dotenv
    load_dotenv()
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print("Note: python-dotenv not installed. Using os.environ directly.")
    print("Install with: pip install python-dotenv")

# Retrieve API keys from environment
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

print("Environment variables loaded:")
print(f"  LANGCHAIN_API_KEY: {'Set' if LANGCHAIN_API_KEY else 'Not set'}")
print(f"  OPENAI_API_KEY: {'Set' if OPENAI_API_KEY else 'Not set'}")
print(f"  HUGGINGFACEHUB_API_TOKEN: {'Set' if HUGGINGFACEHUB_API_TOKEN else 'Not set'}")

# ============================================================
# MOCK IMPLEMENTATIONS FOR DEMONSTRATION (Pure Python/NumPy)
# ============================================================

class Document:
    def __init__(self, page_content, metadata=None):
        self.page_content = page_content
        self.metadata = metadata or {}

class MockEmbeddings:
    def embed_documents(self, texts):
        return [np.random.randn(768).tolist() for _ in texts]
    def embed_query(self, text):
        return np.random.randn(768).tolist()

class MockLLM:
    def invoke(self, prompt):
        class Response:
            content = "Mock response based on context."
        return Response()

class MockVectorStore:
    def __init__(self):
        self.docs = []
    def similarity_search_with_score(self, query, k=5):
        return [(doc, 0.9) for doc in self.docs[:k]]
    def similarity_search(self, query, k=5):
        return self.docs[:k]

# Try importing langchain components, use mocks if not available
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_community.document_loaders import TextLoader
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document as LCDocument
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False
    print("Note: langchain not installed. Using mock implementations.")
    print("Install with: pip install langchain langchain-community")
    # Provide mock classes
    class RecursiveCharacterTextSplitter:
        def __init__(self, chunk_size=1000, chunk_overlap=200):
            self.chunk_size = chunk_size
            self.chunk_overlap = chunk_overlap
        def split_text(self, text):
            # Simple chunking
            chunks = []
            for i in range(0, len(text), self.chunk_size - self.chunk_overlap):
                chunks.append(text[i:i+self.chunk_size])
            return chunks
    
    class TextLoader:
        def __init__(self, path):
            self.path = path
        def load(self):
            with open(self.path) as f:
                content = f.read()
            return [Document(page_content=content)]

    class FAISS:
        @classmethod
        def from_documents(cls, documents, embeddings):
            store = MockVectorStore()
            store.docs = documents
            return store

    class Document:
        def __init__(self, page_content, metadata=None):
            self.page_content = page_content
            self.metadata = metadata or {}

    class MockEmbeddings:
        def embed_documents(self, texts):
            return [np.random.randn(768).tolist() for _ in texts]
        def embed_query(self, text):
            return np.random.randn(768).tolist()

    class MockLLM:
        def invoke(self, prompt):
            class Response:
                content = "Mock response based on context."
            return Response()

    class HuggingFaceEmbeddings:
        def __init__(self, model_name):
            self.model_name = model_name
        def embed_documents(self, texts):
            return [np.random.randn(768).tolist() for _ in texts]
        def embed_query(self, text):
            return np.random.randn(768).tolist()

    class ChatPromptTemplate:
        @classmethod
        def from_template(cls, template):
            return cls(template)
        def __init__(self, template):
            self.template = template
        def __or__(self, other):
            # Simple chain mock
            def chain_func(inputs):
                return f"Prompt: {self.template.format(**inputs)}"
            return chain_func

    class StrOutputParser:
        def __init__(self):
            pass
        def __or__(self, other):
            return other

    # Mock LLM chain
    def mock_llm_chain(inputs):
        return f"Answer based on context: {inputs.get('context', '')[:100]}..."

    llm_chain = mock_llm_chain

    print("Mock langchain components loaded.")

print("\nSetup complete - ready for RAG demonstrations!")

## **1. Basic Naive RAG Pipeline**

**Description:** The simplest RAG architecture. It takes a user query, converts it to an embedding, searches the vector database for top-k documents, and then feeds them to an LLM for generation.

**Pros:** Simple to implement, low latency.
**Cons:** No query rewriting, no reranking. May retrieve irrelevant context if query is not exact match.

---

## **2. Hybrid Search RAG (Vector + Keyword)**

**Description:** Combines vector embeddings for semantic similarity and BM25 (term frequency-inverse document frequency) for keyword matching. Improves retrieval when users ask questions using different words than the source documents.

**Pros:** Captures both conceptual and literal search relevance.
**Cons:** Requires two retrieval engines and merging logic.

---

## **3. RAG with Reranking (Cross-Encoder)**

**Description:** After retrieving initial top-k vectors, a cross-encoder model re-scores the results based on their semantic relevance to the query. This filters out less relevant documents before passing to LLM generation.

**Pros:** Higher precision. Uses dense text similarity models like Cohere Embeddings or Cross-Encoder (BGE).
**Cons:** Adds a slight computational cost and latency.

---

## **4. Query Expansion / Transformative RAG**

**Description:** Instead of feeding the raw query directly to the retriever, an LLM rewrites or expands the query (e.g., splitting a multi-part question into separate sub-queries). Also includes HyDE (Hypothetical Document Embeddings).

**Pros:** Handles complex, ambiguous, or multi-intent questions better.
**Cons:** Requires additional LLM tokens and latency before retrieval.

In [ ]:
from typing import List

class QueryExpansionRAG:
    def __init__(self):
        self.query_expander_llm = None  # Use an LLM like OpenAI or HuggingFace
    
    def expand_query(self, query: str) -> List[str]:
        if not self.query_expander_llm:
            return [query]
        
        prompt = f"Rewrite this query for better search retrieval. Break into multiple parts if needed.\nQuery: {query}\nExpanded Queries:"
        response = self.query_expander_llm.invoke(prompt).content.strip()
        # Split and normalize expanded queries
        sub_queries = [q.strip().strip("'") for q in response.split('\n')]
        return sub_queries
    
    def retrieve_and_combine(self, sub_queries: List[str], k=5) -> str:
        all_retrieved_contexts = ""
        for sub_query in sub_queries:
            docs = FAISS.from_documents(docs_list, embeddings).similarity_search_with_score(sub_query, k=k)
            for doc, _ in docs:
                all_retrieved_contexts += f"\n{doc.page_content}\n"
        
        return all_retrieved_contexts




---

## **5. Agentic RAG (Advanced)**

**Description:** Uses an LLM "agent" to dynamically plan the retrieval strategy based on user questions. The agent can decide whether to search, retrieve, use a tool (e.g., Google Search), or call APIs.

**Pros:** Handles multi-hop reasoning and complex workflows autonomously.
**Cons:** More complex orchestration, higher token costs, potential for loops.

**Example Agentic Workflow Execution**

def run_agent_rag(query):
    """Simulates agentic RAG execution"""
    print(f"User Query: {query}")
    
    # Step 1: Agent plans approach
    print("Agent planning retrieval strategy...")
    plan_result = {"approach": "hybrid_search"}
    
    # Step 2: Execute retrieval
    print("Executing retrieval based on plan...")
    retrieved = vector_store.similarity_search(query, k=5)
    
    # Step 3: Generate answer
    print("Generating final response...")
    context = "\n".join([doc.page_content for doc in retrieved])
    
    return f"Answer generated from {len(retrieved)} documents."

---

## **6. GraphRAG Architecture**

**Description:** Combines knowledge graphs (structured relationships) with vector search. Instead of just storing embeddings, we extract entities and build a graph where nodes represent people/organizations/concepts and edges represent 
relationships. This allows answering relationship-based questions better.

**Pros:** Excellent for entity relationship questions, better at multi-hop reasoning across connected data.
**Cons:** More complex setup, requires graph database integration.

**GraphRAG Query Flow**

def graghg_query(query):
    """
    GraphRAG query execution:
    1. Convert query to embeddings + keywords
    2. Search vector DB for related documents
    3. Traverse knowledge graph for entity relationships
    4. Combine results for generation
    """
    print("GraphRAG Query Processing:")
    print("  1. Embedding query...")
    
    print("  2. Retrieving from vector store...")
    
    print("  3. Traversing knowledge graph relationships...")
    
    print("  4. Generating answer with relationship context...")
    
    return "Answer with entity relationships and provenance."

---

## **7. Multi-Stage / Tiered RAG**

**Description:** Implements a multi-tier approach where simple queries go to a fast cache (e.g., vector index), while complex questions trigger deeper analysis (graph search, agent planning, external tools).

**Pros:** Optimizes performance by routing different query types appropriately.
**Cons:** Requires more infrastructure and careful routing logic.

In [ ]:
class MultiStageRAG:
    def __init__(self):
        self.cache = {}  # Fast lookup cache
        self.vector_store = vector_store
        self.graph_engine = nx.Graph()  # Simplified graph reference
    
    def route_query(self, query):
        """Decides which stage to use based on query complexity"""
        
        # Simple keyword matching first
        if any(keyword in query for keyword in ["who", "when", "where", "what"]):
            return "vector_search"
        
        # Check for relationship questions (multi-hop)
        if any(word in query for word in ["relationship", "between", "from", "to"]):
            return "graph_search"
        
        # Complex or open-ended queries
        return "agent_or_rerank"



---

## **Comparison Table of All RAG Architectures**

This table summarizes all architectures presented in this notebook.

In [ ]:
import pandas as pd

comparison_df = {
    'Architecture': ['Basic Naive RAG', 'Hybrid Search RAG', 
                    'RAG with Reranking', 'Query Expansion RAG', 
                    'Agentic RAG', 'GraphRAG'],
    
    'Complexity': ['Low', 'Medium', 'Medium-High', 'High', 'Very High', 'High'],
    'Latency': ['Fastest', 'Moderate', 'Slower (rerank)', 'Slower (expand)', 
                'Slowest (agent loops)', 'Moderate (graph)'],
    
    'Accuracy': ['Basic', 'Better than Basic', 'High Precision', 
                'Context-Aware', 'Dynamic', 'Relationship-Focused'],
    
    'Best Use Case': ['Simple Q&A', 'Semantic + Keyword', 
                    'High Precision Required', 'Complex Questions', 
                    'Multi-step Reasoning', 'Entity Relationship Queries'],
                    
    'Key Components': ['Vector DB + LLM', 'Vector + BM25', 
                     'Cross-Encoder Reranker', 'LLM Query Rewriter', 
                     'Agent Planner', 'Knowledge Graph + Vector DB']
}

comparison_df = pd.DataFrame(comparison_df)



---

## **Final Recommendations**

Choose your architecture based on your needs:

- **Start with Basic RAG:** If you want a simple implementation.
- **Upgrade to Hybrid Search:** When semantic and keyword search both matter.
- **Add Reranking:** For high-stakes applications requiring precision.
- **Use Query Expansion:** To handle complex user queries better.
- **Consider Agentic RAG:** When you need autonomous decision-making.
- **GraphRAG:** Essential for domain knowledge with relationships (companies, people, products).

---

## **Additional Resources**

1. LangChain: https://js.langchain.com/
2. LlamaIndex: https://docs.llamaindex.ai/
3. Microsoft GraphRAG Blog: https://www.microsoft.com/en-us/research/blog/gragh-a-graph-based-approach-for-retreival-augmented-generation

print("\n" + "="*80)
print("COMPREHENSIVE RAG ARCHITECTURES NOTEBOOK COMPLETED")
print("="*80)

---

# How to Use This Notebook

## Prerequisites

```bash
# Install dependencies
pip install -r requirements.txt

# Optional: For full functionality with real LLMs
pip install langchain langchain-community langchain-huggingface
pip install sentence-transformers faiss-cpu
```

## Environment Setup

1. Copy `.env.example` to `.env`:
   ```bash
   cp .env.example .env
   ```

2. Edit `.env` and add your API keys:
   ```
   LANGCHAIN_API_KEY=your_key_here
   OPENAI_API_KEY=your_key_here
   HUGGINGFACEHUB_API_TOKEN=your_token_here
   ```

## Running the Notebook

1. Start Jupyter:
   ```bash
   jupyter notebook
   ```

2. Open this notebook (`typeOfRags.ipynb`)

3. **Run cells in order** - The first code cell sets up all mock implementations and environment variables

4. Each subsequent code cell demonstrates a different architecture/approach

## What Each Section Does

- **Setup Cell**: Loads environment variables, creates mock classes for demonstration
- **Architecture Cells**: Each cell implements a different RAG approach with runnable code
- **Mock Mode**: By default, runs with mock implementations (no API keys needed)
- **Production Mode**: Install full dependencies and add API keys for real LLM integration

## Customization

To use with real models instead of mocks:

```python
# Replace mock embeddings with real ones
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Replace mock LLM with real one
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Use real vector store
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents, embeddings)
```

## Troubleshooting

- **ModuleNotFoundError**: Run `pip install -r requirements.txt`
- **No API keys**: The notebook runs in mock mode by default
- **Import errors**: Some cells use optional imports (networkx, langgraph, etc.) - install if needed

## Next Steps

- Experiment with different architectures
- Swap mock components for real implementations
- Add your own documents and test queries
- Compare retrieval quality across approaches
